In [ ]:
# NumPy is used for matrix operations and numerical computation.
import numpy as np

# Pandas is used for loading and cleaning the dataset.
import pandas as pd

#Importing the confusion matrix
from sklearn.metrics import confusion_matrix

In [ ]:
# Our dataset (Emal Scam)
file_path = "C:\\Users\\adbar\\Downloads\\MLDataset\\spam_email_dataset.csv"

# Read the CSV file
df = pd.read_csv(file_path)

# Printing the confirmation and basic info here so that we know the dataset loaded correctly.
print("Dataset loaded successfully.")
print("Shape:", df.shape)
print(df.head())

Dataset loaded successfully.
Shape: (10000, 20)
   email_id           subject  \
0         0     Weekly Report   
1         1    Project Update   
2         2    🔥WIN BIG NOW!!   
3         3    🔥WIN BIG NOW!!   
4         4  Meeting Reminder   

                                          email_text  num_words  \
0  budget review - Statement our I claim world st...         19   
1  team sync - President series today already. In...         18   
2  win free urgent offer limited limited urgent u...         19   
3  guarantee click now cash offer click now guara...         16   
4  team sync - Significant property hotel not add...         18   

   num_characters  num_exclamation_marks  num_links  has_suspicious_link  \
0             114                      0          2                    0   
1             114                      0          7                    0   
2             126                      0          4                    1   
3             101                      0      

In [ ]:
#Printing the columns
print("Columns:")
print(df.columns.tolist())

#Printing the data types
print("\nData types:")
print(df.dtypes)

#Printing the missing values just to make sure
print("\nMissing values:")
print(df.isnull().sum())

Columns:
['email_id', 'subject', 'email_text', 'num_words', 'num_characters', 'num_exclamation_marks', 'num_links', 'has_suspicious_link', 'num_attachments', 'has_attachment', 'sender_email', 'sender_domain', 'sender_reputation_score', 'email_hour', 'email_day_of_week', 'is_weekend', 'num_recipients', 'contains_money_terms', 'contains_urgency_terms', 'label']

Data types:
email_id                     int64
subject                     object
email_text                  object
num_words                    int64
num_characters               int64
num_exclamation_marks        int64
num_links                    int64
has_suspicious_link          int64
num_attachments              int64
has_attachment               int64
sender_email                object
sender_domain               object
sender_reputation_score    float64
email_hour                   int64
email_day_of_week            int64
is_weekend                   int64
num_recipients               int64
contains_money_terms         i

In [ ]:
#Our target columns is the label of the data
target_column = "label"
# Drop rows with missing values.
df = df.dropna()

# Separate the raw target column first.
y_raw = df[target_column].copy()

# Separate the raw feature DataFrame.
X_df = df.drop(columns=[target_column]).copy()

X_df = pd.get_dummies(X_df, drop_first=True)

# Convert all feature columns to numeric so NumPy math works cleanly later.
X_df = X_df.astype(float)

print("Feature shape after cleaning:", X_df.shape)
print("Feature columns:")
print(X_df.columns.tolist())

Feature shape after cleaning: (10000, 20027)
Feature columns:
['email_id', 'num_words', 'num_characters', 'num_exclamation_marks', 'num_links', 'has_suspicious_link', 'num_attachments', 'has_attachment', 'sender_reputation_score', 'email_hour', 'email_day_of_week', 'is_weekend', 'num_recipients', 'contains_money_terms', 'contains_urgency_terms', 'subject_Meeting Reminder', 'subject_Project Update', 'subject_Schedule Confirmation', 'subject_URGENT: Cliam your prize', 'subject_Weekly Report', 'subject_You are selected!!!', 'subject_🔥WIN BIG NOW!!', 'email_text_budget review - About east general doctor response west myself system. Interest force method customer.', 'email_text_budget review - Accept beautiful off. Win animal rich know sister story old who. Positive establish threat recent oil.', 'email_text_budget review - According hot beautiful protect available south population. My still company too often at former forward. Pull dark environment best place find shoulder.', 'email_text_b

In [ ]:
# Settinga variable X as the values from X_df.values
X = X_df.values

# Convert the target column into 0/1 values
if y_raw.dtype == object or str(y_raw.dtype).startswith("category") or y_raw.dtype == bool:
    unique_vals = sorted(pd.Series(y_raw).dropna().unique().tolist())

    # Checking that there are exactly 2 unique values
    if len(unique_vals) != 2:
        raise ValueError(
            f"Logistic regression here expects exactly 2 target classes, but found: {unique_vals}"
        )

    target_mapping = {unique_vals[0]: 0, unique_vals[1]: 1}
    y = y_raw.map(target_mapping).astype(int).values

    print("Target mapping used:")
    print(target_mapping)
else:
    y = y_raw.astype(int).values

#Printing the size of the x shape and y shape
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (10000, 20027)
Target vector shape: (10000,)


In [18]:

# Check the unique values in the target vector to confirm they are now 0/1.
unique_targets = np.unique(y)
print("Unique target values:", unique_targets)
if len(unique_targets) != 2 or not np.all(np.isin(unique_targets, [0, 1])):
    raise ValueError(
        f"Target values must be binary 0/1 for this notebook. Found: {unique_targets}"
    )

Unique target values: [0 1]


In [ ]:
#Function that adds a column of ones to the front of X
def add_bias_column(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

In [19]:
#Function that 

def standardize_train_val(X_train, X_val):
    #Calculating the mean
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    # Prevent division by zero for constant columns.
    std[std == 0] = 1

    # Standardize using the training mean and std.
    X_train_scaled = (X_train - mean) / std
    X_val_scaled = (X_val - mean) / std

    return X_train_scaled, X_val_scaled

In [ ]:
#Function that creates the folds
def create_folds(X, y, k=5, seed=42):
    np.random.seed(seed)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    X_shuffled = X[indices]
    y_shuffled = y[indices]

    X_folds = np.array_split(X_shuffled, k)
    y_folds = np.array_split(y_shuffled, k)

    return X_folds, y_folds

In [ ]:
#Converts any real value into 0 or 1's
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def train_logistic_regression(X, y, lam, lr=0.01, epochs=2000):
    n_samples, n_features = X.shape

    weights = np.zeros(n_features)

    for _ in range(epochs):
        z = X @ weights
        predictions = sigmoid(z)

        gradient = (X.T @ (predictions - y)) / n_samples
        regularization = lam * np.r_[0, weights[1:]] / n_samples

        weights -= lr * (gradient + regularization)

    return weights

In [ ]:
#Returning the predicted probabilities for class 1
def predict_proba(X, w):
    return sigmoid(X @ w)

In [ ]:
#Converting the probabilities into class labels
def predict(X, w):
    probs = predict_proba(X, w)
    return (probs >= 0.5).astype(int)

In [ ]:
#Function that will perform k-fold cross validation for logistic regression and return the average classification accuracy
def cross_validate_logistic(X, y, lam, k=5):
    X_folds, y_folds = create_folds(X, y, k=k)
    scores = []

    for i in range(k):
        # Use fold i as the validation set.
        X_val = X_folds[i]
        y_val = y_folds[i]

        # Use all other folds as the training set.
        X_train = np.vstack([X_folds[j] for j in range(k) if j != i])
        y_train = np.hstack([y_folds[j] for j in range(k) if j != i])

        # Standardize correctly using only training statistics.
        X_train_scaled, X_val_scaled = standardize_train_val(X_train, X_val)

        # Add bias column after scaling.
        X_train_scaled = add_bias_column(X_train_scaled)
        X_val_scaled = add_bias_column(X_val_scaled)

        # Train logistic regression model from scratch.
        w = train_logistic_regression(X_train_scaled, y_train, lam)

        # Predict validation classes.
        y_pred = predict(X_val_scaled, w)

        # Classification accuracy.
        accuracy = np.mean(y_pred == y_val)
        scores.append(accuracy)

    return np.mean(scores)

In [ ]:
# Funciton to search over multiple lambdas and return the best lambda, the best score and all of the results in general
def grid_search_logistic(X, y, lambda_values, k=5):
    best_lambda = None

    # Since we want to maximize accuracy, we start with negative infinity.
    best_score = -float("inf")

    results = []

    for lam in lambda_values:
        avg_accuracy = cross_validate_logistic(X, y, lam, k=k)
        results.append((lam, avg_accuracy))

        print(f"Lambda = {lam}, Average CV Accuracy = {avg_accuracy:.4f}")

        if avg_accuracy > best_score:
            best_score = avg_accuracy
            best_lambda = lam

    return best_lambda, best_score, results

In [17]:
lambda_values = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]


best_lambda, best_score, results = grid_search_logistic(X, y, lambda_values, k=5)

print("\nBest lambda:", best_lambda)
print("Best average CV Accuracy:", best_score)

Lambda = 0.0001, Average CV Accuracy = 0.9959
Lambda = 0.001, Average CV Accuracy = 0.9959
Lambda = 0.01, Average CV Accuracy = 0.9959
Lambda = 0.1, Average CV Accuracy = 0.9959
Lambda = 1, Average CV Accuracy = 0.9959
Lambda = 10, Average CV Accuracy = 0.9959
Lambda = 100, Average CV Accuracy = 0.9957
Lambda = 1000, Average CV Accuracy = 0.9900

Best lambda: 0.0001
Best average CV Accuracy: 0.9959


In [20]:
# Printing the results
results_df = pd.DataFrame(results, columns=["Lambda", "Average_CV_Accuracy"])
print(results_df)

      Lambda  Average_CV_Accuracy
0     0.0001               0.9959
1     0.0010               0.9959
2     0.0100               0.9959
3     0.1000               0.9959
4     1.0000               0.9959
5    10.0000               0.9959
6   100.0000               0.9957
7  1000.0000               0.9900


In [21]:
# We are now going to train one final model using the full dataset after we have choose the best lambda using corss validation

# Standardize the full dataset using its own statistics for final training.
full_mean = np.mean(X, axis=0)
full_std = np.std(X, axis=0)
full_std[full_std == 0] = 1

X_scaled = (X - full_mean) / full_std

# Add bias column.
X_scaled = add_bias_column(X_scaled)

# Train the final logistic regression model.
final_weights = train_logistic_regression(X_scaled, y, best_lambda)

In [22]:
y_pred_full = predict(X_scaled, final_weights)
y_prob_full = predict_proba(X_scaled, final_weights)

In [23]:
# Confusion matrix 
cm = confusion_matrix(y, y_pred_full)

# Printing the accuracy (probably helpful )
accuracy = np.mean(y_pred_full == y)

print("Confusion Matrix:")
print(cm)

print("\nFinal training accuracy on full dataset:", accuracy)

Confusion Matrix:
[[6005    0]
 [   0 3995]]

Final training accuracy on full dataset: 1.0


In [24]:
# Printing the class accuracies and the average per class accuracy

tn, fp, fn, tp = cm.ravel()

class_0_accuracy = tn / (tn + fp) if (tn + fp) > 0 else 0
class_1_accuracy = tp / (tp + fn) if (tp + fn) > 0 else 0

average_per_class_accuracy = (class_0_accuracy + class_1_accuracy) / 2

print("Class 0 Accuracy:", class_0_accuracy)
print("Class 1 Accuracy:", class_1_accuracy)
print("Average Per-Class Accuracy:", average_per_class_accuracy)


Class 0 Accuracy: 1.0
Class 1 Accuracy: 1.0
Average Per-Class Accuracy: 1.0
